# 🎤 super_Voz - StyleTTS2 Kaggle Pipeline

Este notebook executa o treinamento completo do StyleTTS2 no Kaggle.

### 🚀 Como usar sem precisar do navegador aberto:
1. Vá em **Settings** e ative a **GPU** (P100 ou T4 x2).
2. Abra o arquivo `styletts2_kaggle_config.yml` no menu lateral e insira suas credenciais do Cloudflare R2.
3. Clique no botão **Save Version** (no topo à direita).
4. Selecione **Save & Run All (Commit)** e clique em **Save**.
5. O Kaggle iniciará o treino em uma sessão separada que dura até 12 horas. Você pode fechar o navegador e voltar depois para pegar os resultados no seu Bucket R2.

In [ ]:
# 🚀 INICIAR TUDO (ONE-CLICK)
import os
import subprocess
from pathlib import Path
import sys

# 1. Configurar Repositório
REPO_URL = "https://github.com/warllemedicao/Voz_styllett2.git"
PROJECT_ROOT = Path("/kaggle/working/Super_voz")

print("--- 1. Preparando Repositório ---")
if PROJECT_ROOT.exists():
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)

PROJECT_DIR = PROJECT_ROOT / "super_Voz"
if not PROJECT_DIR.exists(): PROJECT_DIR = PROJECT_ROOT
os.chdir(PROJECT_DIR)

# 2. Criar Bootstrap
print("--- 2. Criando script de inicialização ---")
pipeline_code = """
import argparse, os, shutil, subprocess, sys, yaml
from pathlib import Path

def run(cmd, cwd=None): 
    print("\\n$ " + " ".join(map(str, cmd)))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    
    with open(args.config, 'r') as f: cfg = yaml.safe_load(f)
    
    # Instalar dependencias basicas
    run([sys.executable, "-m", "pip", "install", "-q", "boto3", "accelerate", "huggingface_hub", "librosa", "soundfile", "phonemizer", "pyyaml"])
    run(["apt-get", "update"], check=False)
    run(["apt-get", "install", "-y", "ffmpeg", "sox", "espeak-ng"], check=False)
    
    script_real = Path("scripts/run_kaggle_styletts2.py")
    if not script_real.exists(): script_real = Path("run_pipeline.py")
    
    if script_real.exists():
        print(f"--- Executando Pipeline: {script_real} ---")
        os.execv(sys.executable, [sys.executable, str(script_real), "--config", args.config])
    else:
        print("❌ ERRO CRÍTICO: Script não encontrado!")
        sys.exit(1)

if __name__ == '__main__': main()
"""
with open("bootstrap_pipeline.py", "w") as f: f.write(pipeline_code.strip())

# 3. Iniciar Pipeline
print("--- 3. Iniciando Treino ---")
subprocess.run([sys.executable, "bootstrap_pipeline.py", "--config", "styletts2_kaggle_config.yml"])